In [4]:
# =====================================================
# Cell 1: System Info & Resource Versions
# =====================================================
# This cell shows basic system info—OS version, GPU, default Python,
# pip, conda (if installed) etc. It's useful for debugging or for
# reproducibility info in a shared notebook.

!set -euxo pipefail

print("=== OS Release Info ===")
!lsb_release -a

print("\n=== GPU Info (nvidia-smi) ===")
!nvidia-smi || echo "[WARNING] nvidia-smi not found"

print("\n=== Default nvcc Version (if available) ===")
!nvcc --version || echo "[WARNING] nvcc not found"

print("\n=== Default Python & Pip Version ===")
!which python
!python --version
!pip --version

print("\n=== Conda Version (if any) ===")
!conda --version || echo "[INFO] 'conda' not yet installed."


/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /bin/bash)
=== OS Release Info ===
/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /bin/bash)
No LSB modules are available.
Distributor ID:	Ubuntu
Description:	Ubuntu 24.04.4 LTS
Release:	24.04
Codename:	noble

=== GPU Info (nvidia-smi) ===
/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /bin/bash)
Sat Nov  7 21:02:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |

In [5]:
# =====================================================
# Cell 2: Install CUDA 11.8
# =====================================================
# Installs CUDA 11.8 from the official NVIDIA repo.
# We pin this version to ensure compatibility with certain libraries.

# Set environment variable for non-interactive installation
import os
os.environ['DEBIAN_FRONTEND'] = 'noninteractive'

print("---- Installing CUDA 11.8 ----")

# STEP 0) Update and install required packages
!sudo apt-get update -y
!sudo apt-get install -y gnupg curl

# STEP 1) Preconfigure keyboard settings to avoid prompts
# You can set it to a default layout, e.g., English (US)
!echo "keyboard-configuration keyboard-configuration/layout select English (US)" | sudo debconf-set-selections
!echo "keyboard-configuration keyboard-configuration/modelcode string pc105" | sudo debconf-set-selections
!echo "keyboard-configuration keyboard-configuration/layoutcode string us" | sudo debconf-set-selections

# STEP 2) Download the cuda-keyring .deb from NVIDIA's repo
!wget https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-keyring_1.1-1_all.deb

# STEP 3) Install the cuda-keyring .deb
!sudo dpkg -i cuda-keyring_1.1-1_all.deb

# STEP 4) Update apt now that the repo & key are configured
!sudo apt-get update -y

# STEP 5) Install CUDA 11.8 without interactive prompts
!sudo apt-get install -y cuda-11-8

# STEP 6) Set environment variables for CUDA
import sys
import subprocess

# Append CUDA paths to environment variables
cuda_path = "/usr/local/cuda-11.8/bin"
cuda_lib = "/usr/local/cuda-11.8/lib64"

# Update PATH and LD_LIBRARY_PATH
os.environ['PATH'] += f":{cuda_path}"
os.environ['LD_LIBRARY_PATH'] = f"{cuda_lib}:" + os.environ.get('LD_LIBRARY_PATH', '')

# Verify CUDA installation
!nvcc --version


---- Installing CUDA 11.8 ----
/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /bin/bash)
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:8 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:9 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Source

In [6]:
# =====================================================
# Cell 3: Double-Check CUDA 11.8 (Optional)
# =====================================================
# In Colab, environment variables may not persist across cells.
# This cell verifies that 'nvcc' is still available and at 11.8.

!export PATH="/usr/local/cuda-11.8/bin:$PATH" && \
nvcc --version


/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /bin/bash)
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


In [7]:
# =====================================================
# Cell 4: Install Miniconda using condacolab
# =====================================================
# We use `condacolab` to install and configure a Miniconda environment
# inside Colab. Once installed, we still need to "source" and activate
# it in bash cells to use conda commands.

!pip install -q condacolab
import condacolab
condacolab.install_miniconda()


/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /bin/bash)

📢 Announcement 📢
condacolab==0.2 will be released soon! Try it with:

    !pip install -q https://github.com/conda-incubator/condacolab/archive/main.zip
    import condacolab
    condacolab.install()

0.2.x introduces a new installation method based on Pixi, with customizable Python versions.
This may be breaking for your workflow. If that's the case, please report it at
https://github.com/conda-incubator/condacolab and pin your `pip install` command to:

    !pip install -q 'condacolab==0.1.*'

✨🍰✨ Everything looks OK!


Miniconda is subject to terms of service: https://anaconda.com/legal/terms/terms-of-service


In [9]:
%%bash
set -ex

# 1. Подключаем conda
source /usr/local/etc/profile.d/conda.sh

# 2. Принимаем лицензию для каналов defaults
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main || true
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r || true

# 3. Настраиваем пути CUDA
export PATH="/usr/local/cuda-11.8/bin:$PATH"
export LD_LIBRARY_PATH="/usr/local/cuda-11.8/lib64:$LD_LIBRARY_PATH"

# 4. Создаем окружение через conda-forge в обход закрытых каналов
conda env remove -y -n trellis || true
conda create -y -n trellis --override-channels -c conda-forge python=3.11
conda activate trellis

# 5. Устанавливаем PyTorch под CUDA 11.8 через pip
pip install torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 --index-url https://download.pytorch.org/whl/cu118

# 6. Фиксируем утилиты сборки и MKL
pip install "mkl<2024.1" "intel-openmp<2024.1" "setuptools<70.0.0" wheel ninja Cython

# 7. Загружаем колеса TRELLIS
pip install https://github.com/Dao-AILab/flash-attention/releases/download/v2.5.8/flash_attn-2.5.8+cu118torch2.3cxx11abiFALSE-cp311-cp311-linux_x86_64.whl
pip install spconv-cu118
pip install kaolin==0.17.0 -f https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.3.0_cu118.html

# 8. Интерфейс Gradio
pip install --force-reinstall "pillow<11.0"
pip install gradio==4.44.1 gradio_litmodel3d==0.0.1

echo "[INFO] trellis environment ready!"


accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r
Jupyter detected...
2 channel Terms of Service accepted
Jupyter detected...
Retrieving notices: - done
Channels:
 - conda-forge
Platform: linux-64
Solving environment: \ | done

## Package Plan ##

  environment location: /usr/local/envs/trellis

  added / updated specs:
    - python=3.11


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _openmp_mutex-4.5          |           20_gnu          28 KB  conda-forge
    bzip2-1.0.8                |      hda65f42_10         252 KB  conda-forge
    ca-certificates-2026.10.23 |       hbd8a1cb_0         127 KB  conda-forge
    ld_impl_linux-64-2.46.1    |default_hbd61a6d_103         725 KB  conda-forge
    libexpat-2.9.0             |       hd2095e1_0          77 KB  conda-forge
    libffi-3.7.0               |    

bash: /usr/local/lib/libtinfo.so.6: no version information available (required by bash)
+ source /usr/local/etc/profile.d/conda.sh
++ export CONDA_EXE=/usr/local/bin/conda
++ CONDA_EXE=/usr/local/bin/conda
++ export _CONDA_EXE=/usr/local/bin/conda
++ _CONDA_EXE=/usr/local/bin/conda
++ export _CE_M=
++ _CE_M=
++ export _CE_CONDA=
++ _CE_CONDA=
++ export CONDA_PYTHON_EXE=/usr/local/bin/python
++ CONDA_PYTHON_EXE=/usr/local/bin/python
++ export _CONDA_ROOT=/usr/local
++ _CONDA_ROOT=/usr/local
++ '[' -z '' ']'
++ export CONDA_SHLVL=0
++ CONDA_SHLVL=0
++ '[' -n '' ']'
++++ dirname /usr/local/bin/conda
+++ dirname /usr/local/bin
++ PATH=/usr/local/condabin:/opt/bin:/usr/local/nvidia/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin:/usr/local/cuda-11.8/bin
++ export PATH
++ '[' -z '' ']'
++ PS1=
+ conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
+ local cmd=tos
+ case "$cmd" in
+ 

In [10]:
# =====================================================
# Cell 6: System Dev Tools & Python Build Libraries
# =====================================================
# Installs system-level development tools: build-essential, cmake, ninja, nvidia-cuda-toolkit
# Then re-activates the 'trellis' environment and upgrades pip, setuptools, wheel, ninja, Cython.

%%bash
set -ex

apt-get update -y
apt-get install -y build-essential cmake ninja-build nvidia-cuda-toolkit

# Make conda available + activate 'trellis'
source /usr/local/etc/profile.d/conda.sh
conda activate trellis

# Upgrade build libraries in Python
pip install --upgrade pip setuptools wheel ninja Cython

echo "[INFO] System dev tools + Python build libs installed."


Hit:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:8 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Reading package lists...
Reading package lists...
Building dependency tree...
Reading state information...
build-essential is already the newest version (12.10ubuntu1).
cmake is already the newest version (3.28.3-1bui

bash: /usr/local/lib/libtinfo.so.6: no version information available (required by bash)
+ apt-get update -y
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
+ apt-get install -y build-essential cmake ninja-build nvidia-cuda-toolkit
+ source /usr/local/etc/profile.d/conda.sh
++ export CONDA_EXE=/usr/local/bin/conda
++ CONDA_EXE=/usr/local/bin/conda
++ export _CONDA_EXE=/usr/local/bin/conda
++ _CONDA_EXE=/usr/local/bin/conda
++ export _CE_M=
++ _CE_M=
++ export _CE_CONDA=
++ _CE_CONDA=
++ export CONDA_PYTHON_EXE=/usr/local/bin/python
++ CONDA_PYTHON_EXE=/usr/local/bin/python
++ export _CONDA_ROOT=/usr/local
++ _CONDA_ROOT=/usr/local
++ '[' -z '' ']'
++ export CONDA_SHLVL=0
++ CONDA_SHLVL=0
++ '[' -n '' ']'
++++ dirname /usr/local/bin/conda
+++ dirname /usr/local/bin
++ PATH=/usr/local/condabin:/opt/bin:/usr/local/nvidia/bin:/usr/local/cuda/bin:/usr/l

May crash at this point. if it does, just resume in the cell below


In [11]:
# =====================================================
# Cell 7: Clone & Setup Microsoft/TRELLIS
# =====================================================
# 1) Source conda script
# 2) Prepend CUDA 11.8
# 3) Activate trellis
# 4) Clean leftover submodule directories
# 5) Clone the MS/TRELLIS repo
# 6) Create 'assets/example_image' folder so app.py won't crash
# 7) Run setup.sh with all relevant flags
# 8) Patch app.py to launch share=True (so you can share the Gradio link)
# 9) Confirm which nvcc is being used

%%bash
set -ex

# Source conda script
source /usr/local/etc/profile.d/conda.sh

# Prepend /usr/local/cuda-11.8 to PATH
export PATH="/usr/local/cuda-11.8/bin:$PATH"
export LD_LIBRARY_PATH="/usr/local/cuda-11.8/lib64:$LD_LIBRARY_PATH"

# Activate 'trellis'
conda activate trellis

# Remove leftover submodule directories
rm -rf /tmp/extensions
rm -rf /tmp/pip-req-build-*

# Clone official MS/TRELLIS repo
rm -rf /content/TRELLIS
git clone --recurse-submodules https://github.com/microsoft/TRELLIS.git /content/TRELLIS
cd /content/TRELLIS
git submodule update --init --recursive

# Create missing folder so app.py won't crash
mkdir -p /content/TRELLIS/assets/example_image

# Run setup.sh
bash ./setup.sh --basic --flash-attn --spconv --mipgaussian --nvdiffrast --kaolin --demo || \
  echo "[WARNING] Some submodules likely failed to build, but that's often normal in Colab."

# Patch app.py => use demo.launch(share=True)
sed -i 's/demo.launch()$/demo.launch(share=True)/' app.py

# Confirm which nvcc is being used now
nvcc --version

echo "[INFO] TRELLIS environment set up; 'app.py' patched."


Submodule path 'trellis/representations/mesh/flexicubes': checked out '815e075a2a400d06c48d94c347674344ed6ae5c5'
[SYSTEM] PyTorch Version: 2.3.0+cu118, CUDA Version: 11.8
  Using cached numpy-2.4.6-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached pillow-13.0.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.0/68.0 MB 62.0 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 MB 50.2 MB/s  0:00:01
Using cached numpy-2.4.6-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.9 MB)
Using cached pillow-13.0.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (7.1 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 90.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 92.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.5/751.5 kB 43.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

bash: /usr/local/lib/libtinfo.so.6: no version information available (required by bash)
+ source /usr/local/etc/profile.d/conda.sh
++ export CONDA_EXE=/usr/local/bin/conda
++ CONDA_EXE=/usr/local/bin/conda
++ export _CONDA_EXE=/usr/local/bin/conda
++ _CONDA_EXE=/usr/local/bin/conda
++ export _CE_M=
++ _CE_M=
++ export _CE_CONDA=
++ _CE_CONDA=
++ export CONDA_PYTHON_EXE=/usr/local/bin/python
++ CONDA_PYTHON_EXE=/usr/local/bin/python
++ export _CONDA_ROOT=/usr/local
++ _CONDA_ROOT=/usr/local
++ '[' -z '' ']'
++ export CONDA_SHLVL=0
++ CONDA_SHLVL=0
++ '[' -n '' ']'
++++ dirname /usr/local/bin/conda
+++ dirname /usr/local/bin
++ PATH=/usr/local/condabin:/opt/bin:/usr/local/nvidia/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin:/usr/local/cuda-11.8/bin
++ export PATH
++ '[' -z '' ']'
++ PS1=
+ export PATH=/usr/local/cuda-11.8/bin:/usr/local/condabin:/opt/bin:/usr/local/nvidia/bin:/usr/local/cuda/bin:/usr/local

You may see warnings in the out of cell below. safe to ignore.

In [12]:
# =====================================================
# Cell 8: Clone & Install mip-splatting
# =====================================================
# 1) Clone the 'mip-splatting' repo if not present
# 2) Install required system packages (libgl1-mesa-dev, etc.)
# 3) Prepend CUDA 11.8
# 4) Activate conda
# 5) Install the diff_gaussian_rasterization submodule
#    which is part of mip-splatting

%%bash
set -ex

cd /content
if [ ! -d "/content/mip-splatting" ]; then
  echo "[INFO] 'mip-splatting' folder not found, cloning now..."
  git clone --recurse-submodules https://github.com/autonomousvision/mip-splatting.git
  cd mip-splatting
  git submodule update --init --recursive
  cd ..
else
  echo "[INFO] 'mip-splatting' folder already exists, skipping clone."
fi

apt-get update -y
apt-get install -y libgl1-mesa-dev build-essential cmake ninja-build

# Prepend /usr/local/cuda-11.8 to PATH & LD_LIBRARY_PATH
export PATH="/usr/local/cuda-11.8/bin:$PATH"
export LD_LIBRARY_PATH="/usr/local/cuda-11.8/lib64:$LD_LIBRARY_PATH"

# Activate 'trellis'
source /usr/local/etc/profile.d/conda.sh
conda activate trellis

# Upgrade Python build tools
pip install --upgrade pip setuptools wheel ninja Cython

# Install diff_gaussian_rasterization from submodule
cd /content/mip-splatting/submodules/diff-gaussian-rasterization
pip install . --no-build-isolation -v

echo "[INFO] diff_gaussian_rasterization installed successfully."


[INFO] 'mip-splatting' folder not found, cloning now...
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:5 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:8 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:10 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Reading package lists...
Reading package lists...
Building dependency tree...
Reading state information...
build-essential is already the newest version (12.10ub

bash: /usr/local/lib/libtinfo.so.6: no version information available (required by bash)
+ cd /content
+ '[' '!' -d /content/mip-splatting ']'
+ echo '[INFO] '\''mip-splatting'\'' folder not found, cloning now...'
+ git clone --recurse-submodules https://github.com/autonomousvision/mip-splatting.git
Cloning into 'mip-splatting'...
+ cd mip-splatting
+ git submodule update --init --recursive
+ cd ..
+ apt-get update -y
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
+ apt-get install -y libgl1-mesa-dev build-essential cmake ninja-build
+ export PATH=/usr/local/cuda-11.8/bin:/opt/bin:/usr/local/nvidia/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin:/usr/local/cuda-11.8/bin
+ PATH=/usr/local/cuda-11.8/bin:/opt/bin:/usr/local/nvidia/bin:/usr/local/cuda/bin:/usr/local/sbin:

CalledProcessError: Command 'b'set -ex\n\ncd /content\nif [ ! -d "/content/mip-splatting" ]; then\n  echo "[INFO] \'mip-splatting\' folder not found, cloning now..."\n  git clone --recurse-submodules https://github.com/autonomousvision/mip-splatting.git\n  cd mip-splatting\n  git submodule update --init --recursive\n  cd ..\nelse\n  echo "[INFO] \'mip-splatting\' folder already exists, skipping clone."\nfi\n\napt-get update -y\napt-get install -y libgl1-mesa-dev build-essential cmake ninja-build\n\n# Prepend /usr/local/cuda-11.8 to PATH & LD_LIBRARY_PATH\nexport PATH="/usr/local/cuda-11.8/bin:$PATH"\nexport LD_LIBRARY_PATH="/usr/local/cuda-11.8/lib64:$LD_LIBRARY_PATH"\n\n# Activate \'trellis\'\nsource /usr/local/etc/profile.d/conda.sh\nconda activate trellis\n\n# Upgrade Python build tools\npip install --upgrade pip setuptools wheel ninja Cython\n\n# Install diff_gaussian_rasterization from submodule\ncd /content/mip-splatting/submodules/diff-gaussian-rasterization\npip install . --no-

**RUN TRELLIS IN NGROK (PROVIDE YOUR NGROK API KEY)**\
if you run the code below, dont forget to edit and put in your nGrok Auth token\
Also, by default, we load kaiser-app.py \
if you want original app.py, uncomment that line and comment out kaiser-app.py


In [ ]:
# =====================================================
# Cell 9: Launch TRELLIS App with Ngrok (show URL after launch)
# =====================================================
# 0) Download kaiser-app.py from your GitHub (if not present)
# 1) Install/Update ngrok if not present
# 2) Configure ngrok with your token (handle new vs. old versions)
# 3) (Re)install Gradio in 'trellis'
# 4) Start ngrok to expose port 7860
# 5) Launch either 'app.py' or 'kaiser-app.py' in 'trellis' environment
# 6) Print ngrok URL AFTER the Gradio app is actually running

import subprocess, time, json, re, os

# ============== STEP 0) Download kaiser-app.py ==============
# We'll place kaiser-app.py in /content/TRELLIS if it's not already there.
TARGET_DIR = "/content/TRELLIS"
kaiser_app_path = os.path.join(TARGET_DIR, "kaiser-app.py")


print("[INFO] Downloading kaiser-app.py from GitHub into /content/TRELLIS ...")
# Note: We use the raw GitHub link to fetch the raw file contents.
# If you change branches or repos, update this URL accordingly.
KAISER_APP_URL = "https://raw.githubusercontent.com/jackel27/Trellis-Colab/main/kaiser-app.py"
subprocess.run([
    "wget", "-q", "-O", kaiser_app_path, KAISER_APP_URL
], check=True)


# ============== USER CHOICE ==============
# Uncomment exactly ONE of the following lines
# to select which app file you want to run.

#APP_FILE = "app.py" # <-- This is TRELLIS Default app.py
APP_FILE = "kaiser-app.py"  # from https://github.com/jackel27/Trellis-Colab/blob/main/kaiser-app.py

# =========================================


#---------------NGROK AUTH TOKEN-----------
# Replace with your own ngrok auth token
NGROK_AUTH = "PLACE NGROK AUTH TOKEN HERE"
#---------------NGROK AUTH TOKEN-----------


# Step 1) Check if ngrok is installed; if not, download & install
try:
    subprocess.run(["which", "ngrok"], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    print("[INFO] ngrok is already installed.")
except subprocess.CalledProcessError:
    print("[INFO] Installing ngrok ...")
    subprocess.run(["wget", "-q", "https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip"], check=True)
    subprocess.run(["unzip", "-o", "ngrok-stable-linux-amd64.zip"], check=True)
    subprocess.run(["chmod", "+x", "ngrok"], check=True)
    subprocess.run(["mv", "ngrok", "/usr/local/bin/ngrok"], check=True)

# (Optional) Print ngrok version to see which command we might need
try:
    print("[INFO] Checking ngrok version...")
    subprocess.run(["ngrok", "version"], check=True)
except:
    print("[WARNING] Could not determine ngrok version.")

# Step 2) Configure ngrok (new vs. old command)
try:
    print("[INFO] Configuring ngrok with new command: 'ngrok config add-authtoken' ...")
    subprocess.run(["ngrok", "config", "add-authtoken", NGROK_AUTH], check=True)
except subprocess.CalledProcessError:
    print("[WARNING] 'ngrok config add-authtoken' failed. Trying older command 'ngrok authtoken' ...")
    subprocess.run(["ngrok", "authtoken", NGROK_AUTH], check=True)

# Step 3) (Re)install Gradio in 'trellis'
print("[INFO] Ensuring Gradio is installed in 'trellis' environment...")
subprocess.run([
    "conda", "run", "-n", "trellis",
    "pip", "install", "gradio==4.44.1", "gradio_litmodel3d==0.0.1"
], check=True)

# Step 4) Start ngrok in background, forwarding port 7860
print("[INFO] Starting ngrok on port 7860 ...")
proc_ngrok = subprocess.Popen(["ngrok", "http", "7860"])

# Give ngrok a few seconds to initialize
time.sleep(4)

# Parse ngrok's local API to get the public URL
public_url = None
try:
    tunnels_json = subprocess.check_output(["curl", "-s", "127.0.0.1:4040/api/tunnels"])
    public_url = json.loads(tunnels_json)["tunnels"][0]["public_url"]
except Exception as e:
    print("[ERROR] Could not get ngrok public URL:", e)

# We'll store the URL now, but only print it after the app is ready
if public_url:
    print("[INFO] Found NGROK URL: ", public_url)
    print("[INFO] Ngrok may not be ready yet; you may need to wait until the app is launched.")
else:
    print("[WARNING] No public URL found from ngrok.")

# Step 5) Launch the selected app file in 'trellis' environment, unbuffered
app_path = os.path.join(TARGET_DIR, APP_FILE)  # e.g. /content/TRELLIS/kaiser-app.py
print(f"[INFO] Launching {app_path} in 'trellis' environment ...")

p = subprocess.Popen(
    ["conda", "run", "-n", "trellis", "python", "-u", app_path],
    cwd=TARGET_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True  # so we can read line by line as strings
)

# Step 6) Stream logs. Only print the final NGROK URL after the app is launched
launch_detected = False

try:
    for line in iter(p.stdout.readline, ""):
        if not line:
            break

        # Print the line to the notebook output
        print(line, end="")

        # If we detect "Running on local URL:", we assume the Gradio app is up
        if "Running on local URL:" in line:
            launch_detected = True
            if public_url:
                print(f"\n[INFO] {APP_FILE} is now live!!! Access via ngrok here:\n{public_url}\n")
            else:
                print("\n[WARNING] We have not found an ngrok URL, but the app claims to be running.\n")

except KeyboardInterrupt:
    print("[INFO] Interrupted by user. Stopping the app.")
    p.kill()
finally:
    p.wait()

if not launch_detected and public_url:
    print("[INFO] The app did not print 'Running on local URL:' but here's the NGROK URL anyway:")
    print(public_url)

print("[INFO] Done. If you want to relaunch, re-run this cell.")


In [16]:
!conda run -n trellis pip install "transformers==4.46.3" "huggingface_hub==0.28.1"


/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /bin/bash)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 84.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 94.2 MB/s  0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.24.7
    Uninstalling huggingface-hub-0.24.7:
      Successfully uninstalled huggingface-hub-0.24.7
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.3
    Uninstalling tokenizers-0.23.3:
      Successfully uninstalled tokenizers-0.23.3
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.2
    Uninstalling transformers-5.22.2:
      Successfully uninstalled transformers-5.22.2

/usr/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /usr/bin/bash)


In [14]:
!conda run -n trellis pip install "huggingface_hub<0.25.0"


/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /bin/bash)
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 2.4.0
    Uninstalling huggingface_hub-2.4.0:
      Successfully uninstalled huggingface_hub-2.4.0
/usr/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /usr/bin/bash)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.22.2 requires huggingface-hub<3.0,>=1.31.0, but you have huggingface-hub 0.24.7 which is incompatible.


**RUN WITH CLOUDFLARE - NO AUTH KEY NEEDED**

In [ ]:
# =====================================================
# Cell 10: Launch TRELLIS App with Cloudflare (show URL after launch)
# =====================================================
# 0) Download kaiser-app.py from your GitHub (if not present)
# 1) Install cloudflared (if not already)
# 2) (Re)install Gradio in 'trellis' environment
# 3) Start a Cloudflare "Quick Tunnel" on localhost:7860
# 4) Launch either 'app.py' or 'kaiser-app.py' in 'trellis' environment
# 5) Print the tunnel URL AFTER the app is actually up.

import subprocess
import time
import re
import os

# ============== STEP 0) Download kaiser-app.py ==============
TARGET_DIR = "/content/TRELLIS"
kaiser_app_path = os.path.join(TARGET_DIR, "kaiser-app.py")

print("[INFO] Downloading kaiser-app.py from GitHub into /content/TRELLIS ...")
KAISER_APP_URL = "https://raw.githubusercontent.com/jackel27/Trellis-Colab/main/kaiser-app.py"
subprocess.run(["wget", "-q", "-O", kaiser_app_path, KAISER_APP_URL], check=True)

# ============== USER CHOICE ==============
# Uncomment exactly ONE of the following lines
# to select which app file you want to run.

#APP_FILE = "app.py"
APP_FILE = "kaiser-app.py"
# =========================================

# Step 1) Install/Update cloudflared via the latest .deb for Linux (amd64)
print("[INFO] Installing/Updating cloudflared...")
try:
    # Download the latest .deb from GitHub
    subprocess.run([
        "wget", "-q", "-O", "cloudflared-linux-amd64.deb",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"
    ], check=True)
    # Attempt dpkg install
    subprocess.run(["sudo", "dpkg", "-i", "cloudflared-linux-amd64.deb"], check=True)
except subprocess.CalledProcessError:
    print("[WARNING] cloudflared may already be installed or dpkg encountered an error.")

# Step 2) (Re)install Gradio in 'trellis' environment
print("[INFO] Ensuring Gradio is installed in 'trellis' environment...")
subprocess.run([
    "conda", "run", "-n", "trellis",
    "pip", "install", "gradio==4.44.1", "gradio_litmodel3d==0.0.1"
], check=True)

# Step 3) Start cloudflared in a subprocess to create a Quick Tunnel from :7860
print("[INFO] Starting cloudflared (Quick Tunnel) on port 7860...")
cloudflare_cmd = ["cloudflared", "tunnel", "--url", "http://127.0.0.1:7860"]
proc_cloudflare = subprocess.Popen(
    cloudflare_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Give cloudflared a few seconds to initialize logs
time.sleep(3)

# We'll store the discovered tunnel URL in `tunnel_url`
tunnel_url = None

def extract_cloudflare_url(log_line: str):
    # We look for "https://xxxxx.trycloudflare.com"
    match = re.search(r"https://.*?\.trycloudflare\.com\S*", log_line)
    return match.group(0) if match else None

print("[INFO] Searching cloudflared logs for the public URL...")

# Check up to 10 seconds for a log line containing the tunnel URL
start_time = time.time()
while (time.time() - start_time) < 10:
    if proc_cloudflare.poll() is not None:
        # cloudflared has exited or failed
        break

    line = proc_cloudflare.stdout.readline()
    if not line:
        time.sleep(0.2)
        continue

    possible_url = extract_cloudflare_url(line)
    if possible_url:
        tunnel_url = possible_url
        break

if tunnel_url:
    print(f"[INFO] Found Cloudflare Tunnel URL => {tunnel_url}")
else:
    print("[WARNING] Could not parse the Cloudflare URL from logs yet.")

# Step 4) Launch either app.py or kaiser-app.py in the 'trellis' environment, unbuffered
app_path = os.path.join(TARGET_DIR, APP_FILE)
print(f"[INFO] Launching TRELLIS app ({APP_FILE}) in the 'trellis' environment...")
p_app = subprocess.Popen(
    ["conda", "run", "-n", "trellis", "python", "-u", app_path],
    cwd=TARGET_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Step 5) Stream logs from the selected app until it prints "Running on local URL:"
launch_detected = False
try:
    for line in iter(p_app.stdout.readline, ""):
        if not line:
            break

        print(line, end="")

        # If the app prints "Running on local URL:", we assume the server is up
        if "Running on local URL:" in line:
            launch_detected = True
            if tunnel_url:
                print(f"\n[INFO] {APP_FILE} is live! Access via Cloudflare:\n{tunnel_url}\n")
            else:
                print("\n[WARNING] We have not found a Cloudflare URL, but the app claims to be running.\n")

except KeyboardInterrupt:
    print("[INFO] Interrupted by user. Stopping the app.")
    p_app.kill()
finally:
    p_app.wait()

# If the app never printed "Running on local URL:", show the URL (if any)
if not launch_detected and tunnel_url:
    print("\n[INFO] The app didn't print 'Running on local URL:' but here's the Cloudflare URL anyway:")
    print(tunnel_url)

print("[INFO] Done. If you'd like to relaunch, just re-run this cell or the entire notebook.")


[INFO] Downloading kaiser-app.py from GitHub into /content/TRELLIS ...
[INFO] Installing/Updating cloudflared...
[INFO] Ensuring Gradio is installed in 'trellis' environment...
[INFO] Starting cloudflared (Quick Tunnel) on port 7860...
[INFO] Searching cloudflared logs for the public URL...
[INFO] Found Cloudflare Tunnel URL => https://magnetic-passive-calls-until.trycloudflare.com
[INFO] Launching TRELLIS app (kaiser-app.py) in the 'trellis' environment...


In [24]:
!rm -rf /content/nvdiffrast
!git clone https://github.com/NVlabs/nvdiffrast.git /content/nvdiffrast
!cp -r /content/nvdiffrast/nvdiffrast /usr/local/envs/trellis/lib/python3.11/site-packages/
!conda run -n trellis python -c "import nvdiffrast.torch as dr; print('[OK] nvdiffrast готов к работе!')"


/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /bin/bash)
/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /bin/bash)
Cloning into '/content/nvdiffrast'...
remote: Enumerating objects: 523, done.
remote: Counting objects: 100% (268/268), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 523 (delta 201), reused 190 (delta 190), pack-reused 255 (from 1)
Receiving objects: 100% (523/523), 11.12 MiB | 17.67 MiB/s, done.
Resolving deltas: 100% (285/285), done.
/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /bin/bash)
/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /bin/bash)
/usr/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /usr/bin/bash)
Traceback (most recent call last):
  File "/usr/local/envs/trellis/lib/python3.11/importlib/metadata/__init__.py", line 563, in from_name
   

In [18]:
!conda run -n trellis pip install "numpy==1.26.4"


/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /bin/bash)
  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.3 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.6
    Uninstalling numpy-2.4.6:
      Successfully uninstalled numpy-2.4.6
/usr/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /usr/bin/bash)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
plyfile 1.1.5 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
rembg 2.0.87 requires numpy<3.0.0,>=2.3.0, but you have numpy 1.2

In [20]:
!conda run -n trellis pip install git+https://github.com/NVlabs/nvdiffrast.git


/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /bin/bash)
  Cloning https://github.com/NVlabs/nvdiffrast.git to /tmp/pip-req-build-mrtbtb2n
  Resolved https://github.com/NVlabs/nvdiffrast.git to commit 253ac4fcea7de5f396371124af597e6cc957bfae
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'
/usr/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /usr/bin/bash)
  Running command git clone --filter=blob:none --quiet https://github.com/NVlabs/nvdiffrast.git /tmp/pip-req-build-mrtbtb2n
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
ERROR: Failed to build 'git+h

In [23]:
!conda run -n trellis pip install --no-build-isolation git+https://github.com/NVlabs/nvdiffrast.git
!conda run -n trellis python -c "import nvdiffrast.torch as dr; print('[OK] nvdiffrast успешно установлен!')"


/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /bin/bash)
  Cloning https://github.com/NVlabs/nvdiffrast.git to /tmp/pip-req-build-9hg53bg7
  Resolved https://github.com/NVlabs/nvdiffrast.git to commit 253ac4fcea7de5f396371124af597e6cc957bfae
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Failed to build nvdiffrast
/usr/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /usr/bin/bash)
  Running command git clone --filter=blob:none --quiet https://github.com/NVlabs/nvdiffrast.git /tmp/pip-req-build-9hg53bg7
  error: subprocess-exited-with-error
  
  × Building wheel for nvdiffrast (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for nvdiffrast
error: failed-wheel-build-for-install

× Failed 

In [25]:
!mkdir -p /usr/local/envs/trellis/lib/python3.11/site-packages/nvdiffrast-0.3.3.dist-info
!printf "Metadata-Version: 2.1\nName: nvdiffrast\nVersion: 0.3.3\n" > /usr/local/envs/trellis/lib/python3.11/site-packages/nvdiffrast-0.3.3.dist-info/METADATA
!sed -i "s/importlib.metadata.version('nvdiffrast')/'0.3.3'/g" /usr/local/envs/trellis/lib/python3.11/site-packages/nvdiffrast/__init__.py 2>/dev/null || true
!conda run -n trellis python -c "import nvdiffrast.torch as dr; print('[OK] nvdiffrast успешно подключен!')"


/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /bin/bash)
/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /bin/bash)
/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /bin/bash)
/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /bin/bash)
/usr/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /usr/bin/bash)
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/usr/local/envs/trellis/lib/python3.11/site-packages/nvdiffrast/torch/__init__.py", line 9, in <module>
    from .ops import RasterizeCudaContext, get_log_level, set_log_level, rasterize, DepthPeeler, interpolate, texture, texture_construct_mip, antialias, antialias_construct_topology_hash, RasterizeGLContext
  File "/usr/local/envs/trellis/lib/python3.11/site-packages/nvdiffrast/torch/ops.py", line 12, in <module>
 

In [27]:
%%bash
set -ex

source /usr/local/etc/profile.d/conda.sh
conda activate trellis

# 1. Отключаем искусственную блокировку версий CUDA внутри PyTorch
python -c "
import torch.utils.cpp_extension as ce
with open(ce.__file__, 'r') as f:
    text = f.read()
    text = text.replace('raise RuntimeError(CUDA_MISMATCH_MESSAGE', 'pass # raise RuntimeError(CUDA_MISMATCH_MESSAGE')
    with open(ce.__file__, 'w') as f:
        f.write(text)
        print('[OK] Блокировка CUDA в PyTorch успешно снята!')
        "

        # 2. Фиксируем архитектуру под видеокарту Tesla T4
        export TORCH_CUDA_ARCH_LIST="7.5"

        # 3. Собираем nvdiffrast (теперь компиляция пойдет без прерывания)
        if [ ! -d "/content/nvdiffrast" ]; then
            git clone https://github.com/NVlabs/nvdiffrast.git /content/nvdiffrast
            fi
            cd /content/nvdiffrast
            pip install --no-build-isolation -v .

            # 4. Сразу собираем модуль гауссианов, чтобы не споткнуться при самом создании 3D-модели
            if [ ! -d "/tmp/mip-splatting" ]; then
                git clone --recurse-submodules https://github.com/autonomousvision/mip-splatting.git /tmp/mip-splatting
                fi
                pip install /tmp/mip-splatting/submodules/diff-gaussian-rasterization --no-build-isolation || true

                # 5. Проверяем готовность модуля
                python -c "import nvdiffrast.torch as dr; print('[УСПЕХ] nvdiffrast скомпилирован и работает!')"


[OK] Блокировка CUDA в PyTorch успешно снята!
Using pip 26.3 from /usr/local/envs/trellis/lib/python3.11/site-packages/pip (python 3.11)
Processing ./.
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for nvdiffrast: filename=nvdiffrast-0.4.0-cp311-cp311-linux_x86_64.whl size=737815 sha256=c1eccae367a4f6494abfea8574313df85cdebe74f43dcadb268b341adfb3423f
  Stored in directory: /tmp/pip-ephem-wheel-cache-2r48lad9/wheels/8f/06/19/2cb18ee7265022df6bacc33139c8d80de59766c11fa43dd2e3
Successfully built nvdiffrast
Processing /tmp/mip-splatting/submodules/diff-gaussian-rasterization
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for diff_gaussian_rasterization: filename=diff_gaussian_rasterization-0.0.0-cp311-cp311-linux_x86_64.whl size=357831 sha256=c9414157a1751336572e92b4b110f09193e8e5507cce90d274b09845e1e51a07
  Stored in directo

bash: /usr/local/lib/libtinfo.so.6: no version information available (required by bash)
+ source /usr/local/etc/profile.d/conda.sh
++ export CONDA_EXE=/usr/local/bin/conda
++ CONDA_EXE=/usr/local/bin/conda
++ export _CONDA_EXE=/usr/local/bin/conda
++ _CONDA_EXE=/usr/local/bin/conda
++ export _CE_M=
++ _CE_M=
++ export _CE_CONDA=
++ _CE_CONDA=
++ export CONDA_PYTHON_EXE=/usr/local/bin/python
++ CONDA_PYTHON_EXE=/usr/local/bin/python
++ export _CONDA_ROOT=/usr/local
++ _CONDA_ROOT=/usr/local
++ '[' -z '' ']'
++ export CONDA_SHLVL=0
++ CONDA_SHLVL=0
++ '[' -n '' ']'
++++ dirname /usr/local/bin/conda
+++ dirname /usr/local/bin
++ PATH=/usr/local/condabin:/opt/bin:/usr/local/nvidia/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin:/usr/local/cuda-11.8/bin
++ export PATH
++ '[' -z '' ']'
++ PS1=
+ conda activate trellis
+ local cmd=activate
+ case "$cmd" in
+ __conda_activate activate trellis
+ '[' -n '' ']'
+ loc

In [29]:
!conda run -n trellis pip install "pydantic==2.10.6"


/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /bin/bash)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 64.0 MB/s  0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.50.4
    Uninstalling pydantic_core-2.50.4:
      Successfully uninstalled pydantic_core-2.50.4
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.14.3
    Uninstalling pydantic-2.14.3:
      Successfully uninstalled pydantic-2.14.3

/usr/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /usr/bin/bash)


In [31]:
!conda run -n trellis pip install "fastapi==0.112.2" "starlette==0.38.2"


/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /bin/bash)
  Attempting uninstall: starlette
    Found existing installation: starlette 1.10.0
    Uninstalling starlette-1.10.0:
      Successfully uninstalled starlette-1.10.0
  Attempting uninstall: fastapi
    Found existing installation: fastapi 0.162.2
    Uninstalling fastapi-0.162.2:
      Successfully uninstalled fastapi-0.162.2

/usr/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /usr/bin/bash)
